# Conversational RAG Video Store Agent

Copyright 2026, Denis Rothman

### 📘 Overview
This notebook demonstrates how to build a **Conversational Video RAG Agent** using **Oracle 23ai**.

Unlike traditional video search (which relies on filenames), this agent uses **Vector Search** to understand the *content* of a video. It retrieves specific video segments based on the semantic meaning of your query (e.g., *"Show me a player scoring a goal"*) and plays the exact clip directly in the notebook.

### 🚀 Workflow
1.  **Ingest:** The notebook acts as a "crawler" to fetch video metadata (CSVs) from the remote GitHub repository.
2.  **Vectorize:** It generates embeddings for every video comment using OpenAI and stores them in the **Oracle 23ai Hybrid Vector Database**.
3.  **Retrieve:** The `VideoStoreAgent` performs a vector similarity search to find the most relevant video timestamps.
4.  **Generate:** An LLM synthesizes the results into a helpful answer and displays the video clip starting at the precise moment of action.

---

# Installation and Setup

## Installing OpenAI

In [ ]:
#!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.24.0

In [ ]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"


## Global Configuration Flags

This section acts as the **Control Panel** for the notebook. These boolean flags determine which administrative tasks will execute during this run. This allows the DBA to run the notebook safely in "maintenance mode" without accidentally recreating tables or wiping data.

* **`unzip_wallet`**: Set to `True` only for the initial setup to extract the Oracle Wallet configuration. Once extracted, set to `False` to avoid redundant file operations.
* **`create_tables`**: Set to `True` to execute DDL statements. This initializes the `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE` tables. (Keep `False` if tables already exist).
* **`empty_tables`**: Set to `True` to perform a **TRUNCATE** operation. **Warning:** This will permanently delete all vector data from the tables while preserving the structure.

In [ ]:
unzip_wallet=False

In [ ]:
unzip_wallet=False  # True to unzip the wallet. False to only unzip the wallet once

## Oracle environment Setup & Wallet Extraction

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if unzip_wallet==True:
  !unzip -o "/content/drive/MyDrive/files/oracle/Wallet_*.zip" -d "/content/drive/MyDrive/files/oracle"

## Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [ ]:
!pip install oracledb==3.4.1

In [ ]:
import oracledb
print(oracledb.__version__)

## Additional imports

In [ ]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

## Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [ ]:
import os
from google.colab import userdata
oracle_dsn = userdata.get('oracle_dsn')

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

# Step 1: The "Crawler" - Ingesting Metadata & URLs (Idempotent / No Duplicates)

In [ ]:
# --- Oracle DBA Console: Synchronized Schema Provisioning ---
def provision_chapter_9_registry(cursor):
    # 1. Define the Catalog Table (Matches Crawler Column Names)
    assets_sql = """
    CREATE TABLE MEDIA_ASSETS (
        asset_id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
        filename VARCHAR2(255),
        category VARCHAR2(100),
        video_url VARCHAR2(1000)
    )
    """

    # 2. Define the Vector Store (Matches Crawler Column Names)
    segments_sql = """
    CREATE TABLE MEDIA_SEGMENTS (
        segment_id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
        asset_id NUMBER,
        timestamp_start VARCHAR2(50),
        description CLOB,
        description_vector VECTOR(1536, FLOAT32), -- Pinned for text-embedding-3-small
        CONSTRAINT fk_asset FOREIGN KEY (asset_id) REFERENCES MEDIA_ASSETS(asset_id)
    )
    """

    # 3. Idempotent Execution Loop
    schemas = {
        "MEDIA_ASSETS": assets_sql,
        "MEDIA_SEGMENTS": segments_sql
    }

    print("--- 🛠️ DBA Console: Provisioning Chapter 9 Registry ---")
    for table_name, create_query in schemas.items():
        cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table_name}'")
        if cursor.fetchone()[0] == 0:
            print(f"Creating table: {table_name}...")
            cursor.execute(create_query)
            print(f"✅ {table_name} provisioned successfully.")
        else:
            print(f"✔️ {table_name} already exists.")
    print("--- Registry Ready for Ingestion ---\n")

# Run the provisioning
provision_chapter_9_registry(cursor)

In [ ]:
# -------------------------------------------------------------------------
# Step 1: The "Crawler" - Ingesting Metadata & URLs (Idempotent / No Duplicates)
# -------------------------------------------------------------------------
import pandas as pd
import requests
import io
from openai import OpenAI
import os
from google.colab import userdata
import array

# --- 1. Authentication & Setup ---
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input=[text], model=model).data[0].embedding

# --- 2. Configuration Maps ---
GITHUB_API_BASE = "https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/main/Chapter09/comments/"
VIDEO_BASE_URL = "https://raw.githubusercontent.com/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/main/Chapter09/videos/"

video_files = [
    "alpinist1.mp4", "ball_passing_goal.mp4", "basketball1.mp4", "basketball2.mp4",
    "basketball3.mp4", "basketball4.mp4", "basketball5.mp4", "female_player_after_scoring.mp4",
    "football1.mp4", "football2.mp4", "hockey1.mp4", "jogging1.mp4", "jogging2.mp4",
    "skiing1.mp4", "soccer_pass.mp4", "soccer_player_head.mp4", "soccer_player_running.mp4",
    "surfer1.mp4", "surfer2.mp4", "swimming1.mp4", "walking1.mp4"
]

def ingest_video_metadata(cursor, connection):
    print(f"--- 🚀 Starting Metadata Ingestion for {len(video_files)} Assets ---")

    headers = {}

    total_segments = 0

    for video_file in video_files:
        print(f"\nProcessing: {video_file}...")

        # --- 1. IDEMPOTENCY CHECK ---
        # Check if this file is already in the catalog
        check_sql = "SELECT asset_id FROM MEDIA_ASSETS WHERE filename = :1"
        cursor.execute(check_sql, [video_file])
        existing_record = cursor.fetchone()

        if existing_record:
            print(f"   ⚠️ Asset already exists (ID: {existing_record[0]}). Skipping ingestion.")
            continue
        # ----------------------------

        csv_filename = video_file + ".csv"
        api_url = GITHUB_API_BASE + csv_filename

        # A. Fetch the CSV (Text Data Only)
        response = requests.get(api_url, headers=headers)
        if response.status_code != 200:
            print(f"   ⚠️ Metadata CSV not found. Skipping.")
            continue

        # B. Insert Catalog Entry
        category = ''.join([i for i in video_file if not i.isdigit()]).replace('.mp4', '').capitalize().replace('_', ' ')
        video_url = VIDEO_BASE_URL + video_file

        out_id_var = cursor.var(int)

        try:
            cursor.execute(
                "INSERT INTO MEDIA_ASSETS (filename, category, video_url) VALUES (:1, :2, :3) RETURNING asset_id INTO :4",
                [video_file, category, video_url, out_id_var]
            )
            asset_id = out_id_var.getvalue()
            if isinstance(asset_id, list): asset_id = asset_id[0]
            print(f"   ✅ Cataloged URL (ID: {asset_id})")

            # C. Process Text Descriptions
            df = pd.read_csv(io.StringIO(response.text))
            df.columns = [c.strip().lower() for c in df.columns]

            timestamp_col = next((c for c in df.columns if c in ['framenumber', 'frame', 'timestamp']), None)
            comment_col = next((c for c in df.columns if c in ['comment', 'description']), None)

            if not timestamp_col or not comment_col:
                print(f"   ❌ CSV format error. Skipping.")
                continue

            rows_to_insert = []
            for index, row in df.iterrows():
                timestamp = str(row[timestamp_col])
                comment = str(row[comment_col])

                vector_data = get_embedding(comment)
                vector_array = array.array('f', vector_data)

                rows_to_insert.append((asset_id, timestamp, comment, vector_array))

            if rows_to_insert:
                cursor.executemany(
                    """
                    INSERT INTO MEDIA_SEGMENTS
                    (asset_id, timestamp_start, description, description_vector)
                    VALUES (:1, :2, :3, :4)
                    """,
                    rows_to_insert
                )

            print(f"   🧠 Stored {len(rows_to_insert)} text segments.")
            connection.commit()
            total_segments += len(rows_to_insert)

        except Exception as e:
            print(f"   ❌ Error: {e}")
            connection.rollback()

    print(f"\n--- ✅ Ingestion Complete. Total Text Segments: {total_segments} ---")

if 'cursor' in globals() and 'connection' in globals():
    ingest_video_metadata(cursor, connection)
else:
    print("❌ Database connection not active.")

# Step 2: The Video Store Agent (Class Definition & Instantiation)

In [ ]:
# -------------------------------------------------------------------------
# Step 2: The Video Store Agent (Class Definition & Instantiation)
# -------------------------------------------------------------------------
import array
import base64
import requests
import os
from IPython.display import display, HTML

class VideoStoreAgent:
    def __init__(self, client, cursor):
        self.client = client
        self.cursor = cursor
        self.system_prompt = """
        You are an expert Video Library Assistant.
        1. Analyze the user's request.
        2. Use the database search results to answer.
        3. Always cite the filename and timestamp.
        4. Be concise and helpful.
        """

    def get_embedding(self, text):
        text = text.replace("\n", " ")
        vec = self.client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding
        return array.array('f', vec)

    def search_videos(self, query, top_k=1):
        """ Retrieves the top match from Oracle """
        print(f"🔍 Searching database for: '{query}'...")
        query_vector = self.get_embedding(query)

        sql = """
        SELECT
            a.filename,
            a.video_url,
            s.timestamp_start,
            s.description
        FROM MEDIA_SEGMENTS s
        JOIN MEDIA_ASSETS a ON s.asset_id = a.asset_id
        ORDER BY VECTOR_DISTANCE(s.description_vector, :1) ASC
        FETCH FIRST :2 ROWS ONLY
        """

        self.cursor.execute(sql, [query_vector, top_k])
        return self.cursor.fetchall()

    def display_video_clip(self, url, filename, timestamp):
        """
        Downloads and displays video using Base64.
        Robustly handles timestamps in 'MM:SS' or raw seconds format.
        """
        print(f"⬇️ Downloading video to memory: {filename}...")

        try:
            # 1. Download Video
            headers = {}

            response = requests.get(url, headers=headers)
            if response.status_code != 200:
                print(f"❌ Failed to download video (Status {response.status_code})")
                return

            # 2. Encode to Base64
            video_b64 = base64.b64encode(response.content).decode()

            # 3. Calculate Seconds (Robust Logic)
            try:
                if ':' in str(timestamp):
                    parts = str(timestamp).split(':')
                    seconds = int(parts[0]) * 60 + int(parts[1])
                else:
                    seconds = int(float(timestamp))
            except ValueError:
                seconds = 0

            # 4. Display
            html_code = f"""
            <div style="background-color:#1e1e1e; padding:15px; border-radius:10px; margin-top:10px; width: 600px; font-family: sans-serif; color: white;">
                <div style="border-bottom: 1px solid #444; padding-bottom: 5px; margin-bottom: 10px;">
                    <b>🎬 Match Found: {filename}</b>
                </div>
                <div style="color: #4CAF50; margin-bottom: 5px;">
                    👉 Auto-seeking to: <b>{timestamp}</b> (Starts at {seconds}s)
                </div>
                <video width="100%" controls autoplay>
                    <source src="data:video/mp4;base64,{video_b64}#t={seconds}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
            </div>
            """
            display(HTML(html_code))

        except Exception as e:
            print(f"❌ Error processing video: {e}")

    def ask(self, user_query):
        # 1. Retrieval
        results = self.search_videos(user_query, top_k=1)

        if not results:
            return "I couldn't find any matching videos."

        top_match = results[0]

        # 2. Generation
        context = f"File: {top_match[0]}, Time: {top_match[2]}, Desc: {top_match[3]}"
        prompt = f"User: {user_query}\nContext: {context}"

        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt}
            ]
        )

        # 3. Output
        print("\n🤖 Agent Response:")
        print("-" * 50)
        print(response.choices[0].message.content)
        print("-" * 50)

        # 4. Display
        self.display_video_clip(top_match[1], top_match[0], top_match[2])

# --- Instantiate the Agent ---
# This is the line that defines 'agent'
agent = VideoStoreAgent(client, cursor)
print("✅ Agent is ready to chat!")

## Interacting with the Agent

In [ ]:
# Interacting with the Agent
agent.ask("Find me a clip of a basketball player scoring a basket")

In [ ]:
# Interacting with the Agent
agent.ask("I want to see a surfer")

In [ ]:
# Interacting with the Agent
agent.ask("I want to see a soccer player")